# `detect_toc` + `extract_native_toc` — PDF table of contents

Module under test: [`src/docpipeline/parsing/pdf/toc/`](../../../../src/docpipeline/parsing/pdf/toc/)

| Function | Strategy | Output |
|---|---|---|
| `detect_toc(pdf)` | Heuristic scoring (5 signals, ZERO LLM) | `TocDetectionResult` — `has_toc`, `confidence`, `toc_pages`, `reason` |
| `extract_native_toc(pdf)` | Native bookmarks via PyMuPDF | `pd.DataFrame` — `level`, `title`, `page`, `indicator` |
| `extract_toc_from_links(pdf)` | Internal hyperlinks | `pd.DataFrame` — `text`, `page_num` |
| `extract_toc_dotted(pdf)` | Dotted leader lines | `pd.DataFrame` — `text`, `page_num`, `source_page` |

Demo on `data/CG contrats MRH/CG MRH_GMF.pdf`.

In [1]:
import sys
!{sys.executable} -m pip install -q -e ..

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from docpipeline.parsing.pdf.toc import (
    detect_toc,
    has_native_toc,
    extract_native_toc,
    extract_toc_from_links,
    extract_toc_dotted,
    nest_toc,
)

pd.set_option('display.max_colwidth', 100)

PDF = Path('../data/CG contrats MRH/CG MRH_GMF.pdf')

# ── 1. Heuristic detection ────────────────────────────────────────────────────
result = detect_toc(PDF)
display(Markdown('### 1. `detect_toc` — heuristic result'))
display(pd.DataFrame([result.to_dict()]))

# ── 2. Native bookmarks ───────────────────────────────────────────────────────
display(Markdown(f'### 2. `extract_native_toc` — has_native_toc={has_native_toc(PDF)}'))
toc_df = extract_native_toc(PDF)
display(toc_df if not toc_df.empty else pd.DataFrame({'result': ['no native bookmarks']}))

# ── 3. Link-based extraction ──────────────────────────────────────────────────
display(Markdown('### 3. `extract_toc_from_links`'))
links_df = extract_toc_from_links(PDF, max_pages=10)
display(links_df.head(15) if not links_df.empty else pd.DataFrame({'result': ['no internal links']}))

# ── 4. Dotted leader extraction ───────────────────────────────────────────────
display(Markdown('### 4. `extract_toc_dotted`'))
dotted_df = extract_toc_dotted(PDF)
display(dotted_df.head(15) if not dotted_df.empty else pd.DataFrame({'result': ['no dotted leaders']}))